# LLM/RAG Initialization

In [ ]:
import pickle

# Load data for the RAG (processed_data.pkl)
with open('../data/processed_data.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

In [ ]:
from langchain import PromptTemplate, LLMChain, HuggingFacePipeline
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import re
from rag_source import *
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from huggingface_hub import InferenceClient
from huggingface_hub import InferenceApi
import json

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  #This is directory

# Initialize client with your Hugging Face token
from langchain.llms import HuggingFaceEndpoint

llm = InferenceClient(
    model="meta-llama/Llama-3.1-70B",
    token=""
)


# Define a PromptTemplate that accepts prompt_text as an input
prompt_template = PromptTemplate(
    input_variables=["prompt_text"],
    template="{prompt_text}"
)

# Initialize LLMChain with the LLM and prompt template
llm_chain = LLMChain(
    prompt=prompt_template,
    llm=llm
)

# Function to directly analyze the provided text prompt
def analyze_text_prompt(query, processed_data):
    # Pass the prompt text to LLMChain as a dictionary

    prompt_text = process_query(query, processed_data, embedding_model) + " The correct answer is, "

    inputs = {
        "prompt_text": prompt_text
    }
    
    # Run the LLM chain and get the result
    result = llm_chain.run(inputs)

    return prompt_text, result


In [ ]:
from langchain.llms.base import LLM
from typing import Optional, List, Mapping, Any
import requests

class HuggingFaceEndpointLLM(LLM):
    endpoint_url: str
    api_key: str

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        headers = {
            # "Accept": "application/json",
            "Authorization": f"Bearer {self.api_key}",
            # "Content-Type": "application/json"
        }

        payload = {
            "inputs": prompt,
            "parameters": {
                "max_new_tokens": 250,
                "temperature": 0.7,
                "top_p": 0.9
            }
        }

        response = requests.post(self.endpoint_url, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()[0]["generated_text"]

    @property
    def _llm_type(self) -> str:
        return "custom-huggingface-endpoint"

llm = HuggingFaceEndpointLLM(
    endpoint_url="https://tqnarz7dcpw8ve6r.us-east-1.aws.endpoints.huggingface.cloud",
    api_key=""  # Replace with your actual key
)

# Function to directly analyze the provided text prompt
def analyze_text_prompt(query, processed_data):
    # Pass the prompt text to LLMChain as a dictionary

    prompt_text = process_query(query, processed_data, embedding_model) + " The correct answer is, "

    inputs = {
        "prompt_text": prompt_text
    }
    
    # Run the LLM chain and get the result
    result = llm._call(prompt_text)

    return prompt_text, result

# Pass the query

In [ ]:
from docx import Document
import pandas as pd
import openai
import os

# Set your OpenAI key (or assume it's set via env var)
# openai.api_key = os.getenv("OPENAI_API_KEY")

# === STEP 1: Load Q&A from DOCX ===
def load_questions_from_docx(file_path):
    doc = Document(file_path)
    qna_pairs = []

    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for i in range(0, len(paragraphs) - 1, 2):  # assuming Q on even lines, A on odd
        question = paragraphs[i]
        expected = paragraphs[i + 1]
        qna_pairs.append((question, expected))
    
    return qna_pairs

# === STEP 2: Query GPT ===
def ask_gpt(question): #, model="gpt-4"):
    try:
        # Use the analyze_text_prompt function to get the prompt and response
        prompt, response = analyze_text_prompt(question, loaded_data)
        # Remove the prompt_text from the beginning of result
        if response.startswith(prompt):
            response = response[len(prompt):]
        return prompt, response
    except Exception as e:
        return question, f"[ERROR] {str(e)}"

# === STEP 3: Run All ===
def process_questions(docx_path, output_csv = f"../data/{type(llm_chain.llm).__name__}_eval_output.csv"):
    qna_pairs = load_questions_from_docx(docx_path)

    results = []
    for idx, (question, expected) in enumerate(qna_pairs, 1):
        print(f"Processing {idx}/{len(qna_pairs)}...")
        prompt, model_answer = ask_gpt(question)
        results.append({
            "Question": question,
            "Expected Answer": expected,
            "Model Answer": model_answer,
            "Prompt": prompt,
            "Evaluation": ""  # Leave blank for manual input
        })

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Done. Saved results to {output_csv}")

# === USAGE ===
# Replace with your actual DOCX file
process_questions("../data/Questions_Emergence.docx")

# file = pd.read_csv(f"../data/{type(llm_chain.llm).__name__}_eval_output.csv")
# safe_model_name = llm.repo_id.replace("/", "_")
# output_csv = f"../data/{safe_model_name}_eval_output.csv"
# file.to_csv(output_csv)